In [ ]:
from __future__ import annotations
import copy
import os
import time
from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

import warnings

torch.manual_seed(42)
np.random.seed(42)

In [ ]:
warnings.filterwarnings("ignore")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
IMAGE_SIZE = 224
IMAGE_MEAN = [0.485, 0.456, 0.406]
IMAGE_STD = [0.229, 0.224, 0.225]

BLOCKS_NUM = [2, 2, 2, 2]
PATIENCE = 5
MIN_DELTA = 0.001
VAL_BATCH_SIZE = 128

DRY_RUN = False
BASE_LR = 0.001
BASE_BATCH_SIZE = 32
BASE_EPOCHS = 20
BEST_CHECKPOINT_PATH = Path.cwd() / "best_model_v3_grid.pth"

TRAIN_NUM_WORKERS = min(4, max(2, (os.cpu_count() or 2) // 2))
EVAL_NUM_WORKERS = min(2, TRAIN_NUM_WORKERS)
PIN_MEMORY = device.type == "cuda"
PERSISTENT_WORKERS = TRAIN_NUM_WORKERS > 0
PREFETCH_FACTOR = 2 if TRAIN_NUM_WORKERS > 0 else None
AMP_ENABLED = device.type == "cuda"
CHANNELS_LAST = device.type == "cuda"
CUDA_BENCHMARK = device.type == "cuda"

SAVE_BEST_CHECKPOINT = False

torch.backends.cudnn.benchmark = CUDA_BENCHMARK

GRID_LR = [BASE_LR / 2, BASE_LR, BASE_LR * 2]
GRID_BATCH_SIZE = [16, 32, 48]
GRID_EPOCHS = [max(2, BASE_EPOCHS // 2), BASE_EPOCHS, BASE_EPOCHS + max(5, BASE_EPOCHS // 2)]

DRY_GRID_LR = [BASE_LR]
DRY_GRID_BATCH_SIZE = [BASE_BATCH_SIZE]
DRY_GRID_EPOCHS = [2]

In [ ]:
criterion = nn.CrossEntropyLoss()

In [ ]:
food_dir = Path("/kaggle/input/datasets/bloodlaac/products-dataset/v3")

# food_dir = Path("/Users/bloodlaac/Projects/univer/graduation/datasets/v3")

FOOD_CLASSES = ['Fresh', 'Bad']
CLASS_TO_IDX = {name: idx for idx, name in enumerate(FOOD_CLASSES)}

In [ ]:
class LabeledDataset(Dataset):
    def __init__(
        self,
        food_dir: Path,
        food_classes: list[str],
        transform=None,
    ) -> None:

        self.food_dir = food_dir
        self.food_classes = food_classes
        self.transform = transform
        self.images_paths = []
        self.labels = []
        self.classes = list(food_classes)
        self.class_to_idx = {name: idx for idx, name in enumerate(food_classes)}

        for cls_name in food_classes:
            class_path = Path(food_dir)
            class_path /= cls_name

            for image_path in sorted(class_path.iterdir()):
                if image_path.is_file():
                    self.images_paths.append(image_path)
                    self.labels.append(self.class_to_idx[cls_name])

    def __len__(self) -> int:
        return len(self.images_paths)

    def __getitem__(self, index: int):
        image = Image.open(self.images_paths[index]).convert("RGB")
        label = self.labels[index]

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
from torchvision.transforms import InterpolationMode

FILL = tuple(int(255 * x) for x in IMAGE_MEAN)

train_transforms = T.Compose([
    T.Resize(
        int(IMAGE_SIZE * 1.15),
        interpolation=InterpolationMode.BILINEAR,
    ),
    T.RandomResizedCrop(
        IMAGE_SIZE,
        scale=(0.85, 1.00),
        ratio=(0.95, 1.05),
        interpolation=InterpolationMode.BILINEAR,
        antialias=True,
    ),
    T.RandomHorizontalFlip(p=0.3),
    T.ColorJitter(
        brightness=0.12,
        contrast=0.12,
        saturation=0.08,
        hue=0.02,
    ),
    T.RandomAffine(
        degrees=0,
        translate=(0.04, 0.04),
        scale=(0.98, 1.02),
        shear=(-2, 2),
        interpolation=InterpolationMode.BILINEAR,
        fill=FILL,
    ),
    T.ToTensor(),
    T.Normalize(IMAGE_MEAN, IMAGE_STD),
])

In [ ]:
eval_transforms = T.Compose([
    T.Resize(
        (IMAGE_SIZE, IMAGE_SIZE),
        interpolation=InterpolationMode.BILINEAR,
    ),
    T.ToTensor(),
    T.Normalize(IMAGE_MEAN, IMAGE_STD),
])

In [ ]:
generator = torch.Generator().manual_seed(42)

base_dataset = LabeledDataset(food_dir, FOOD_CLASSES, transform=None)
dataset_len = len(base_dataset)

train_size = int(0.6 * dataset_len)
val_size = int(0.2 * dataset_len)
test_size = dataset_len - train_size - val_size

indices = torch.randperm(dataset_len, generator=generator).tolist()

train_indices = indices[:train_size]
val_indices = indices[train_size:train_size + val_size]
test_indices = indices[train_size + val_size:]

In [ ]:
train_full_dataset = LabeledDataset(
    food_dir,
    FOOD_CLASSES,
    transform=train_transforms
)

val_full_dataset = LabeledDataset(
    food_dir,
    FOOD_CLASSES,
    transform=eval_transforms
)

test_full_dataset = LabeledDataset(
    food_dir,
    FOOD_CLASSES,
    transform=eval_transforms
)

train_dataset = torch.utils.data.Subset(train_full_dataset, train_indices)
val_dataset = torch.utils.data.Subset(val_full_dataset, val_indices)
test_dataset = torch.utils.data.Subset(test_full_dataset, test_indices)

In [ ]:
def build_dataloader(dataset, batch_size: int, shuffle: bool, num_workers: int, drop_last: bool):
    loader_kwargs = {
        "dataset": dataset,
        "batch_size": batch_size,
        "shuffle": shuffle,
        "num_workers": num_workers,
        "pin_memory": PIN_MEMORY,
        "drop_last": drop_last,
    }

    if num_workers > 0:
        loader_kwargs["persistent_workers"] = PERSISTENT_WORKERS
        loader_kwargs["prefetch_factor"] = PREFETCH_FACTOR

    return DataLoader(**loader_kwargs)


def build_dataloaders_for_batch_size(batch_size: int):
    if batch_size <= 0:
        raise ValueError("batch_size must be positive")

    eval_batch_size = max(VAL_BATCH_SIZE, batch_size)

    train_dataloader = build_dataloader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=TRAIN_NUM_WORKERS,
        drop_last=True,
    )
    val_dataloader = build_dataloader(
        val_dataset,
        batch_size=eval_batch_size,
        shuffle=False,
        num_workers=EVAL_NUM_WORKERS,
        drop_last=False,
    )
    test_dataloader = build_dataloader(
        test_dataset,
        batch_size=eval_batch_size,
        shuffle=False,
        num_workers=EVAL_NUM_WORKERS,
        drop_last=False,
    )

    return train_dataloader, val_dataloader, test_dataloader

In [ ]:
class Block(nn.Module):
    """
    Create basic unit of ResNet.

    Consists of two convolutional layers.

    """

    def __init__(
            self,
            in_channels: int,
            out_channels: int,
            stride: int = 1,
            downsampling=None
        ) -> None:

        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=3,
            stride=stride,
            padding=1
        )
        self.bn1 = nn.BatchNorm2d(num_features=out_channels)
        self.bn2 = nn.BatchNorm2d(num_features=out_channels)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv2d(
            in_channels=out_channels,
            out_channels=out_channels,
            kernel_size=3,
            stride=1,  # TODO: Replace with padding="same"
            padding=1
        )
        self.downsampling = downsampling

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        input = x

        pred = self.bn1(self.conv1(x))
        pred = self.relu(pred)
        pred = self.bn2(self.conv2(pred))

        if self.downsampling is not None:
            input = self.downsampling(x)

        pred += input
        pred = self.relu(pred)

        return pred

In [ ]:
class ResNet(nn.Module):
    """
    Build model ResNet and return prediction

    """

    def __init__(self, blocks_num_list: list[int]) -> None:
        """
        ResNet init.

        Parameters
        ----------
        blocks_num_list : list[int]
                          Number of basic blocks for each layer.

        """
        super().__init__()

        self.in_channels = 64  # Default number of channels for first layer. Mutable!

        # Reduce resolution of picture by 2
        # 224 -> 112
        self.conv1 = nn.Conv2d(
            in_channels=3,
            out_channels=64,
            kernel_size=7,
            stride=2,
            padding=3
        )
        self.batch_norm = nn.BatchNorm2d(64)
        self.relu = nn.ReLU()
        self.pooling = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)  # 112 -> 56

        self.layer1 = self.create_layer(  # Default stride. No resolution reduction.
            out_channels=64,
            num_blocks=blocks_num_list[0]
        )
        self.layer2 = self.create_layer(  # Resolution reduction. 56 -> 28
            out_channels=128,
            num_blocks=blocks_num_list[1],
            stride=2
        )
        self.layer3 = self.create_layer(  # Resolution reduction. 28 -> 14
            out_channels=256,
            num_blocks=blocks_num_list[2],
            stride=2
        )
        self.layer4 = self.create_layer(  # Resolution reduction. 14 -> 7
            out_channels=512,
            num_blocks=blocks_num_list[3],
            stride=2
        )

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, len(FOOD_CLASSES))

    def create_layer(
            self,
            out_channels: int,
            num_blocks: int,
            stride: int = 1
        ) -> nn.Sequential:
        """
        Create ResNet layer.

        Parameters
        ----------
        out_channels : int
            Number of output channels per block
        num_blocks : int
            Number of blocks per layer
        stride : int, default=1
            Step of filter in conv layer

        """
        downsampling = None

        if stride != 1:
            downsampling = nn.Sequential(
                nn.Conv2d(
                    in_channels=self.in_channels,
                    out_channels=out_channels,
                    kernel_size=1,
                    stride=stride
                ),
                nn.BatchNorm2d(out_channels)
            )

        blocks: list[Block] = []

        blocks.append(Block(
            in_channels=self.in_channels,
            out_channels=out_channels,
            stride=stride,
            downsampling=downsampling
        ))

        self.in_channels = out_channels

        for _ in range(num_blocks - 1):
            blocks.append(Block(out_channels, out_channels))

        return nn.Sequential(*blocks)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        pred = self.batch_norm(self.conv1(x))
        pred = self.relu(pred)
        pred = self.pooling(pred)

        pred = self.layer1(pred)
        pred = self.layer2(pred)
        pred = self.layer3(pred)
        pred = self.layer4(pred)

        pred = self.avgpool(pred)
        pred = torch.flatten(pred, 1)
        pred = self.fc(pred)

        return pred

In [ ]:
def build_param_grid(dry_run: bool = False):
    learning_rates = DRY_GRID_LR if dry_run else GRID_LR
    batch_sizes = DRY_GRID_BATCH_SIZE if dry_run else GRID_BATCH_SIZE
    epochs_list = DRY_GRID_EPOCHS if dry_run else GRID_EPOCHS

    learning_rates = list(dict.fromkeys(learning_rates))
    batch_sizes = list(dict.fromkeys(batch_sizes))
    epochs_list = list(dict.fromkeys(epochs_list))

    return [
        {
            "run_id": run_id,
            "learning_rate": learning_rate,
            "batch_size": batch_size,
            "epochs_num": epochs_num,
            "dry_run": dry_run,
        }
        for run_id, (learning_rate, batch_size, epochs_num) in enumerate(
            product(learning_rates, batch_sizes, epochs_list),
            start=1,
        )
    ]


def plot_run_history(result):
    epochs_ran = result["epochs_ran"]
    epochs_axis = np.arange(1, epochs_ran + 1)
    config = result["config"]
    title_suffix = (
        f"run {config['run_id']} | lr={config['learning_rate']}, "
        f"batch={config['batch_size']}, epochs={config['epochs_num']}"
    )

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

    ax1.plot(epochs_axis, result["train_accuracy_history"], label="Train accuracy")
    ax1.plot(epochs_axis, result["val_accuracy_history"], label="Validation accuracy")
    ax2.plot(epochs_axis, result["train_loss_history"], label="Train loss")
    ax2.plot(epochs_axis, result["val_loss_history"], label="Validation loss")

    ax1.set_title(f"Accuracy history | {title_suffix}")
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel("Accuracy")
    ax1.legend()
    ax1.grid(True)

    ax2.set_title(f"Loss history | {title_suffix}")
    ax2.set_xlabel("Epochs")
    ax2.set_ylabel("Loss")
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.show()


def plot_run_test_metrics(result):
    config = result["config"]
    title_suffix = (
        f"run {config['run_id']} | lr={config['learning_rate']}, "
        f"batch={config['batch_size']}, epochs={config['epochs_num']}"
    )

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

    ax1.bar(["Test accuracy"], [result["test_accuracy"]], color="#2e8b57")
    ax1.set_ylim(0, 1)
    ax1.set_title(f"Test accuracy | {title_suffix}")
    ax1.grid(True, axis="y")

    ax2.bar(["Test loss"], [result["test_loss"]], color="#c44e52")
    ax2.set_title(f"Test loss | {title_suffix}")
    ax2.grid(True, axis="y")

    plt.tight_layout()
    plt.show()


def is_better_result(candidate, current_best):
    if current_best is None:
        return True

    candidate_acc = candidate["test_accuracy"]
    best_acc = current_best["test_accuracy"]

    if candidate_acc > best_acc and not np.isclose(candidate_acc, best_acc):
        return True

    if np.isclose(candidate_acc, best_acc):
        return candidate["test_loss"] < current_best["test_loss"]

    return False


In [ ]:
def move_images_to_device(images: torch.Tensor) -> torch.Tensor:
    images = images.to(device, non_blocking=True)

    if CHANNELS_LAST:
        images = images.contiguous(memory_format=torch.channels_last)

    return images


def validate(model, loader, criterion):
    correct, total = 0, 0
    val_loss = 0.0
    val_start_time = time.perf_counter()

    model.eval()

    with torch.no_grad():
        for images, labels in loader:
            images = move_images_to_device(images)
            labels = labels.to(device, non_blocking=True)

            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=AMP_ENABLED,
            ):
                logits = model(images)
                loss = criterion(logits, labels)

            val_loss += loss.item() * len(labels)
            total += len(labels)

            predictions = torch.argmax(logits, dim=1)
            correct += (predictions == labels).sum().item()

    accuracy = correct / total
    loss = val_loss / total
    val_seconds = time.perf_counter() - val_start_time
    val_samples_per_sec = total / max(val_seconds, 1e-6)

    return accuracy, loss, val_samples_per_sec

In [ ]:
def train(
    model,
    criterion,
    train_loader,
    val_loader,
    optimizer,
    epochs=10,
    patience=5,
    min_delta=0.0,
):
    train_acc_history, train_loss_history = [], []
    val_acc_history, val_loss_history = [], []
    epoch_seconds = []
    train_samples_per_sec_history = []
    val_samples_per_sec_history = []

    best_val_loss = float("inf")
    best_model_state = copy.deepcopy(model.state_dict())
    epochs_without_improvement = 0
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)

    for epoch in tqdm(range(epochs), leave=False):
        model.train()

        correct, total = 0, 0
        epoch_loss = 0.0
        epoch_start_time = time.perf_counter()

        for step, (images, labels) in enumerate(train_loader, start=1):
            images = move_images_to_device(images)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=AMP_ENABLED,
            ):
                logits = model(images)
                loss = criterion(logits, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            predictions = torch.argmax(logits, dim=1)

            total += len(labels)
            epoch_loss += loss.item() * len(labels)
            correct += (predictions == labels).sum().item()
            accuracy = correct / total

            if step % 100 == 0:
                temp_loss = epoch_loss / total

                print(
                    f"Epoch: [{epoch + 1}/{epochs}], Step: [{step}/{len(train_loader)}]\n"
                    f"Train loss: {temp_loss:.4f}, Train Accuracy: {accuracy:.4f}\n"
                )

        train_seconds = time.perf_counter() - epoch_start_time
        train_samples_per_sec = total / max(train_seconds, 1e-6)
        train_acc_history.append(accuracy)
        train_loss_history.append(epoch_loss / total)
        epoch_seconds.append(train_seconds)
        train_samples_per_sec_history.append(train_samples_per_sec)

        val_acc, val_loss, val_samples_per_sec = validate(model, val_loader, criterion)
        val_acc_history.append(val_acc)
        val_loss_history.append(val_loss)
        val_samples_per_sec_history.append(val_samples_per_sec)

        print(
            f"Epoch: [{epoch + 1}/{epochs}] has passed\n"
            f"Train loss: {train_loss_history[-1]:.4f}, Train accuracy: {train_acc_history[-1]:.4f}\n"
            f"Validation loss: {val_loss:.4f}, Validation accuracy: {val_acc:.4f}\n"
            f"Epoch seconds: {train_seconds:.2f}, Train samples/sec: {train_samples_per_sec:.2f}, "
            f"Validation samples/sec: {val_samples_per_sec:.2f}\n"
        )

        if val_loss < best_val_loss - min_delta:
            best_val_loss = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            print(f"Early stopping after epoch {epoch + 1}")
            break

    model.load_state_dict(best_model_state)

    max_memory_allocated_mb = None
    if device.type == "cuda":
        max_memory_allocated_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2)

    return {
        "train_accuracy_history": train_acc_history,
        "train_loss_history": train_loss_history,
        "val_accuracy_history": val_acc_history,
        "val_loss_history": val_loss_history,
        "train_accuracy": train_acc_history[-1],
        "train_loss": train_loss_history[-1],
        "val_accuracy": val_acc_history[-1],
        "val_loss": val_loss_history[-1],
        "best_model_state": best_model_state,
        "epochs_ran": len(train_acc_history),
        "epoch_seconds": epoch_seconds,
        "train_samples_per_sec_history": train_samples_per_sec_history,
        "val_samples_per_sec_history": val_samples_per_sec_history,
        "train_samples_per_sec": train_samples_per_sec_history[-1],
        "val_samples_per_sec": val_samples_per_sec_history[-1],
        "max_memory_allocated_mb": max_memory_allocated_mb,
    }

In [ ]:
def test(model, loader, criterion):
    model.eval()
    test_loss = 0.0
    correct = 0
    total = 0
    y_true, y_pred = [], []

    with torch.no_grad():
        for images, labels in loader:
            images = move_images_to_device(images)
            labels = labels.to(device, non_blocking=True)

            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=AMP_ENABLED,
            ):
                logits = model(images)
                loss = criterion(logits, labels)

            test_loss += loss.item() * len(labels)
            total += len(labels)

            preds = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()

            y_true.extend(labels.cpu().tolist())
            y_pred.extend(preds.cpu().tolist())

    avg_loss = test_loss / total
    accuracy = correct / total
    cm = confusion_matrix(y_true, y_pred)
    report = classification_report(y_true, y_pred, digits=4)

    return accuracy, avg_loss, cm, report


def save_best_checkpoint(result, checkpoint_path: Path):
    checkpoint = {
        "model_state": result["best_model_state"],
        "classes": FOOD_CLASSES,
        "class_to_idx": CLASS_TO_IDX,
        "num_classes": len(FOOD_CLASSES),
        "training_params": {
            "learning_rate": result["config"]["learning_rate"],
            "batch_size": result["config"]["batch_size"],
            "epochs_num": result["config"]["epochs_num"],
            "dry_run": result["config"]["dry_run"],
        },
        "metrics": {
            "train_accuracy": result["train_accuracy"],
            "train_loss": result["train_loss"],
            "val_accuracy": result["val_accuracy"],
            "val_loss": result["val_loss"],
            "test_accuracy": result["test_accuracy"],
            "test_loss": result["test_loss"],
            "epoch_seconds": result["epoch_seconds"],
            "train_samples_per_sec": result["train_samples_per_sec"],
            "val_samples_per_sec": result["val_samples_per_sec"],
            "max_memory_allocated_mb": result["max_memory_allocated_mb"],
        },
    }

    torch.save(checkpoint, checkpoint_path)
    return checkpoint_path


def load_model_from_state(model_state):
    model = ResNet(BLOCKS_NUM).to(device)
    if CHANNELS_LAST:
        model = model.to(memory_format=torch.channels_last)
    model.load_state_dict(model_state)
    model.eval()
    return model


def run_single_experiment(config):
    print(
        f"Run {config['run_id']}: lr={config['learning_rate']}, "
        f"batch_size={config['batch_size']}, epochs={config['epochs_num']}"
    )

    model = ResNet(BLOCKS_NUM).to(device)
    if CHANNELS_LAST:
        model = model.to(memory_format=torch.channels_last)

    optimizer = optim.SGD(model.parameters(), lr=config["learning_rate"], momentum=0.9)
    train_dataloader, val_dataloader, test_dataloader = build_dataloaders_for_batch_size(
        config["batch_size"]
    )

    train_result = train(
        model,
        criterion,
        train_dataloader,
        val_dataloader,
        optimizer=optimizer,
        epochs=config["epochs_num"],
        patience=PATIENCE,
        min_delta=MIN_DELTA,
    )

    test_accuracy, test_loss, cm, report = test(model, test_dataloader, criterion)

    return {
        "config": config,
        "train_accuracy_history": train_result["train_accuracy_history"],
        "train_loss_history": train_result["train_loss_history"],
        "val_accuracy_history": train_result["val_accuracy_history"],
        "val_loss_history": train_result["val_loss_history"],
        "train_accuracy": train_result["train_accuracy"],
        "train_loss": train_result["train_loss"],
        "val_accuracy": train_result["val_accuracy"],
        "val_loss": train_result["val_loss"],
        "test_accuracy": test_accuracy,
        "test_loss": test_loss,
        "confusion_matrix": cm,
        "classification_report": report,
        "best_model_state": train_result["best_model_state"],
        "epochs_ran": train_result["epochs_ran"],
        "epoch_seconds": train_result["epoch_seconds"],
        "train_samples_per_sec_history": train_result["train_samples_per_sec_history"],
        "val_samples_per_sec_history": train_result["val_samples_per_sec_history"],
        "train_samples_per_sec": train_result["train_samples_per_sec"],
        "val_samples_per_sec": train_result["val_samples_per_sec"],
        "max_memory_allocated_mb": train_result["max_memory_allocated_mb"],
    }

In [ ]:
def run_grid_search(dry_run: bool = False):
    param_grid = build_param_grid(dry_run=dry_run)
    results = []
    best_result = None
    best_checkpoint_path = None

    mode_label = "DRY_RUN" if dry_run else "FULL_RUN"
    print(f"Starting {mode_label} grid search with {len(param_grid)} runs\n")

    for config in param_grid:
        result = run_single_experiment(config)
        plot_run_history(result)
        plot_run_test_metrics(result)

        summary_result = {
            key: value
            for key, value in result.items()
            if key != "best_model_state"
        }
        results.append(summary_result)

        if is_better_result(result, best_result):
            if SAVE_BEST_CHECKPOINT:
                best_checkpoint_path = save_best_checkpoint(result, BEST_CHECKPOINT_PATH)
                checkpoint_path_str = str(best_checkpoint_path)
            else:
                best_checkpoint_path = None
                checkpoint_path_str = None

            best_result = summary_result | {
                "best_model_state": result["best_model_state"],
                "checkpoint_path": checkpoint_path_str,
            }

    if best_result is None:
        raise RuntimeError("Grid search did not produce any results")

    best_run_id = best_result["config"]["run_id"]
    records = []

    for result in results:
        row = {
            "run_id": result["config"]["run_id"],
            "learning_rate": result["config"]["learning_rate"],
            "batch_size": result["config"]["batch_size"],
            "epochs_num": result["config"]["epochs_num"],
            "train_accuracy": result["train_accuracy"],
            "train_loss": result["train_loss"],
            "val_accuracy": result["val_accuracy"],
            "val_loss": result["val_loss"],
            "test_accuracy": result["test_accuracy"],
            "test_loss": result["test_loss"],
            "epoch_seconds": result["epoch_seconds"][-1],
            "train_samples_per_sec": result["train_samples_per_sec"],
            "val_samples_per_sec": result["val_samples_per_sec"],
            "max_memory_allocated_mb": result["max_memory_allocated_mb"],
            "checkpoint_path": str(best_checkpoint_path) if result["config"]["run_id"] == best_run_id else None,
            "is_best": result["config"]["run_id"] == best_run_id,
        }
        records.append(row)

    results_df = pd.DataFrame(records).sort_values(
        by=["test_accuracy", "test_loss"],
        ascending=[False, True],
    ).reset_index(drop=True)

    best_model = load_model_from_state(best_result["best_model_state"])
    return results, best_result, results_df, best_model, best_checkpoint_path

In [ ]:
results, best_result, results_df, best_model, best_checkpoint_path = run_grid_search(
    dry_run=DRY_RUN
)


In [ ]:
print("Grid search summary:")
print(results_df.to_string(index=False))

print(f"\nBest checkpoint saved to: {best_checkpoint_path or 'disabled'}")
print("Best parameters:")
print(best_result["config"])
print("Best metrics:")
print(
    {
        "train_accuracy": round(best_result["train_accuracy"], 4),
        "train_loss": round(best_result["train_loss"], 4),
        "val_accuracy": round(best_result["val_accuracy"], 4),
        "val_loss": round(best_result["val_loss"], 4),
        "test_accuracy": round(best_result["test_accuracy"], 4),
        "test_loss": round(best_result["test_loss"], 4),
        "epoch_seconds": [round(value, 2) for value in best_result["epoch_seconds"]],
        "train_samples_per_sec": round(best_result["train_samples_per_sec"], 2),
        "val_samples_per_sec": round(best_result["val_samples_per_sec"], 2),
        "max_memory_allocated_mb": None
        if best_result["max_memory_allocated_mb"] is None
        else round(best_result["max_memory_allocated_mb"], 2),
    }
)
print("Classification report:")
print(best_result["classification_report"])

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

disp = ConfusionMatrixDisplay(
    confusion_matrix=best_result["confusion_matrix"],
    display_labels=FOOD_CLASSES,
)
disp.plot(ax=ax, cmap=plt.cm.Blues, colorbar=False)
ax.set_title("Best Run Confusion Matrix")

plt.tight_layout()
plt.show()

In [ ]:
results_df

In [ ]:
best_model

In [ ]:
def predict_image(model, image_path: str | Path):
    image = Image.open(image_path).convert("RGB")
    x = eval_transforms(image).unsqueeze(0)
    x = move_images_to_device(x)

    with torch.no_grad():
        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=AMP_ENABLED,
        ):
            logits = model(x)
            probs = F.softmax(logits, dim=1)

    pred_idx = torch.argmax(probs, dim=1).item()
    pred_class = FOOD_CLASSES[pred_idx]
    pred_conf = probs[0, pred_idx].item()

    all_probs = {
        FOOD_CLASSES[i]: float(probs[0, i].item())
        for i in range(len(FOOD_CLASSES))
    }

    return {
        "predicted_class": pred_class,
        "confidence": pred_conf,
        "probabilities": all_probs,
    }

In [ ]:
image_path = "/Users/bloodlaac/Projects/univer/graduation/datasets/v3/Fresh/FreshApple (121).jpg"

result = predict_image(best_model, image_path)

print("Предсказание:", result["predicted_class"])
print("Уверенность:", round(result["confidence"], 4))
print("Вероятности:", result["probabilities"])

In [ ]:
image = Image.open(image_path).convert("RGB")
plt.imshow(image)